# PathSnap - Reproducible Candidate Ranking Model Training

This notebook trains and compares exactly five scikit-learn models:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Gradient Boosting
5. K-Nearest Neighbors

Training uses trip-level `GroupKFold`; the separate
`data/processed/student_test_candidates.csv` file is never used for fitting or
hyperparameter selection. The repository marks label-quality rows with
`aligned == 1`, so only aligned rows are fitted and evaluated; excluded rows
are reported.

Run every cell from top to bottom in Google Colab.

In [1]:
from pathlib import Path

print("Current directory:", Path.cwd())

print("Training CSV exists:",
      Path("data/processed/train_candidates.csv").exists())

print("Student test CSV exists:",
      Path("data/processed/student_test_candidates.csv").exists())

Current directory: c:\Users\SHREYASH M YADWAD\Downloads
Training CSV exists: False
Student test CSV exists: False


In [2]:
import os
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\SHREYASH M YADWAD\Desktop\PathSnap"
)

os.chdir(PROJECT_DIR)

print("Current directory:", Path.cwd())

Current directory: C:\Users\SHREYASH M YADWAD\Desktop\PathSnap


In [3]:
import json, warnings, joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, log_loss, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
np.random.seed(42)

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
except Exception:
    OPTUNA_AVAILABLE = False

print("Optuna available:", OPTUNA_AVAILABLE)


Optuna available: False


## 1. Load CSV files

The notebook first looks for repository paths, then offers Colab upload.
It can also load from Google Drive using the optional cell below.

In [4]:
def find_repo_csv(filename):
    candidates = [
        Path("/content/data/processed") / filename,
        Path("data/processed") / filename,
        Path("/content") / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    return None

TRAIN_PATH = find_repo_csv("train_candidates.csv")
TEST_PATH = find_repo_csv("student_test_candidates.csv")

if TRAIN_PATH is None:
    try:
        from google.colab import files
        print("Upload train_candidates.csv")
        uploaded = files.upload()
        TRAIN_PATH = Path(next(iter(uploaded)))
    except ImportError:
        raise FileNotFoundError("train_candidates.csv was not found")

if TEST_PATH is None:
    try:
        from google.colab import files
        print("Upload student_test_candidates.csv")
        uploaded = files.upload()
        TEST_PATH = Path(next(iter(uploaded)))
    except ImportError:
        raise FileNotFoundError("student_test_candidates.csv was not found")

train_full = pd.read_csv(TRAIN_PATH)
test_full = pd.read_csv(TEST_PATH)
print("Training CSV:", TRAIN_PATH, train_full.shape)
print("Student test CSV:", TEST_PATH, test_full.shape)


Training CSV: data\processed\train_candidates.csv (1084912, 18)
Student test CSV: data\processed\student_test_candidates.csv (113648, 18)


In [8]:
# Optional Google Drive loading:
# from google.colab import drive
# drive.mount("/content/drive")
# TRAIN_PATH = Path("/content/drive/MyDrive/PathSnap/data/processed/train_candidates.csv")
# TEST_PATH = Path("/content/drive/MyDrive/PathSnap/data/processed/student_test_candidates.csv")
# train_full = pd.read_csv(TRAIN_PATH)
# test_full = pd.read_csv(TEST_PATH)


## 2. Complete data audit

The audit reports duplicates, missing/infinite values, candidate counts,
label quality, constant features, and non-numeric columns. Invalid label
counts are reported and not silently discarded.

In [5]:
REQUIRED = {
    "trip_id", "obs_idx", "timestamp", "candidate_arc_id", "distance_m",
    "heading_diff", "speed_mps", "bearing", "is_match", "projected_offset_m",
    "arc_length_m", "remaining_distance_m", "offset_ratio", "time_delta_s",
    "gps_displacement_m", "candidate_heading_deg", "heading_missing", "aligned"
}
missing_required = sorted(REQUIRED - set(train_full.columns))
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

print("=== TRAINING DATA AUDIT ===")
print("Number of rows:", len(train_full))
print("Number of columns:", train_full.shape[1])
print("Column names:", train_full.columns.tolist())
print("\nData types:")
print(train_full.dtypes.to_string())
print("\nMissing values:")
print(train_full.isna().sum().to_string())
print("Duplicate rows:", int(train_full.duplicated().sum()))
print("Duplicate candidate identities:", int(
    train_full.duplicated(["trip_id", "obs_idx", "candidate_arc_id"]).sum()
))
print("Unique trip count:", train_full["trip_id"].nunique())
print("Unique observation count:", train_full.groupby(["trip_id", "obs_idx"]).ngroups)
print("Candidate count per observation:")
print(train_full.groupby(["trip_id", "obs_idx"]).size().describe().to_string())
print("Positive and negative class counts:", train_full["is_match"].value_counts().to_dict())
print("Positive class percentage:", float(train_full["is_match"].mean() * 100))

full_labels = train_full.groupby(["trip_id", "obs_idx"])["is_match"].sum()
print("Observations with exactly one positive:", int((full_labels == 1).sum()))
print("Observations with zero positives:", int((full_labels == 0).sum()))
print("Observations with multiple positives:", int((full_labels > 1).sum()))

numeric = train_full.select_dtypes(include=[np.number])
print("Constant numeric features:", [
    c for c in numeric.columns if train_full[c].nunique(dropna=False) <= 1
])
print("Non-numeric columns:", train_full.select_dtypes(exclude=[np.number]).columns.tolist())
print("Infinite values:", int(np.isinf(numeric.to_numpy(dtype=float)).sum()))

train = train_full.loc[train_full["aligned"] == 1].copy().reset_index(drop=True)
print("\nAligned rows retained:", len(train), "of", len(train_full))
print("Unaligned rows excluded:", int((train_full["aligned"] != 1).sum()))

train_labels = train.groupby(["trip_id", "obs_idx"])["is_match"].sum()
print("Aligned observations with exactly one positive:", int((train_labels == 1).sum()))
print("Aligned observations with zero positives:", int((train_labels == 0).sum()))
print("Aligned observations with multiple positives:", int((train_labels > 1).sum()))
print("Aligned duplicate candidate identities:", int(
    train.duplicated(["trip_id", "obs_idx", "candidate_arc_id"]).sum()
))


=== TRAINING DATA AUDIT ===
Number of rows: 1084912
Number of columns: 18
Column names: ['trip_id', 'obs_idx', 'timestamp', 'candidate_arc_id', 'distance_m', 'heading_diff', 'speed_mps', 'bearing', 'is_match', 'projected_offset_m', 'arc_length_m', 'remaining_distance_m', 'offset_ratio', 'time_delta_s', 'gps_displacement_m', 'candidate_heading_deg', 'heading_missing', 'aligned']

Data types:
trip_id                    int64
obs_idx                    int64
timestamp                float64
candidate_arc_id           int64
distance_m               float64
heading_diff             float64
speed_mps                float64
bearing                  float64
is_match                   int64
projected_offset_m       float64
arc_length_m             float64
remaining_distance_m     float64
offset_ratio             float64
time_delta_s             float64
gps_displacement_m       float64
candidate_heading_deg    float64
heading_missing            int64
aligned                    int64

Missing val

## 3. Features and leakage protection

Only predictive geometric, directional, and movement features are included.
Trip, observation, candidate, timestamp, target, and alignment columns are
excluded from the feature matrix.

In [6]:
FEATURE_COLUMNS = [
    "distance_m", "heading_diff", "speed_mps", "bearing",
    "projected_offset_m", "arc_length_m", "remaining_distance_m",
    "offset_ratio", "time_delta_s", "gps_displacement_m",
    "candidate_heading_deg", "heading_missing"
]
TARGET = "is_match"
IDENTIFIERS = ["trip_id", "obs_idx", "candidate_arc_id"]

for name, df in [("train", train), ("student_test", test_full)]:
    missing = [c for c in FEATURE_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing features: {missing}")

train_test = test_full.loc[test_full["aligned"] == 1].copy().reset_index(drop=True)
X = train[FEATURE_COLUMNS].copy()
y = train[TARGET].astype(int)
groups = train["trip_id"].astype(str)
X_test = train_test[FEATURE_COLUMNS].copy()
y_test = train_test[TARGET].astype(int)

leakage = set(FEATURE_COLUMNS).intersection(
    {TARGET, "trip_id", "obs_idx", "candidate_arc_id", "timestamp", "aligned"}
)
if leakage:
    raise AssertionError(f"Target or identifier leakage: {sorted(leakage)}")

print("Feature matrix:", X.shape)
print("Features:", FEATURE_COLUMNS)
print("Training trips:", groups.nunique())
print("Student test aligned rows:", len(train_test))


Feature matrix: (1049122, 12)
Features: ['distance_m', 'heading_diff', 'speed_mps', 'bearing', 'projected_offset_m', 'arc_length_m', 'remaining_distance_m', 'offset_ratio', 'time_delta_s', 'gps_displacement_m', 'candidate_heading_deg', 'heading_missing']
Training trips: 80
Student test aligned rows: 113220


## 4. Shared five-fold trip GroupKFold

Five folds are used if 80 trips are available and the folds preserve both
classes. Every model receives exactly these same fold indices.

In [7]:
unique_trips = groups.nunique()
n_splits = 5 if unique_trips >= 5 else min(unique_trips, 3)
if n_splits < 2:
    raise ValueError("At least two training trips are required")

cv = GroupKFold(n_splits=n_splits)
cv_splits = list(cv.split(X, y, groups=groups))

for fold, (train_idx, valid_idx) in enumerate(cv_splits, 1):
    train_trips = set(groups.iloc[train_idx])
    valid_trips = set(groups.iloc[valid_idx])
    overlap = train_trips & valid_trips
    print(
        f"Fold {fold}: train={len(train_idx)} rows/{len(train_trips)} trips, "
        f"valid={len(valid_idx)} rows/{len(valid_trips)} trips, "
        f"train classes={np.bincount(y.iloc[train_idx], minlength=2).tolist()}, "
        f"valid classes={np.bincount(y.iloc[valid_idx], minlength=2).tolist()}, "
        f"overlap={len(overlap)}"
    )
    if overlap:
        raise AssertionError("Trip leakage between folds")


Fold 1: train=839755 rows/65 trips, valid=209367 rows/15 trips, train classes=[694579, 145176], valid classes=[170744, 38623], overlap=0
Fold 2: train=839573 rows/64 trips, valid=209549 rows/16 trips, train classes=[687498, 152075], valid classes=[177825, 31724], overlap=0
Fold 3: train=839705 rows/64 trips, valid=209417 rows/16 trips, train classes=[699249, 140456], valid classes=[166074, 43343], overlap=0
Fold 4: train=839808 rows/64 trips, valid=209314 rows/16 trips, train classes=[691847, 147961], valid classes=[173476, 35838], overlap=0
Fold 5: train=837647 rows/63 trips, valid=211475 rows/17 trips, train classes=[688119, 149528], valid classes=[177204, 34271], overlap=0


## 5. Pipelines and evaluation utilities

Preprocessing is fitted only inside each fold. Raw positive-class probabilities
are used for ranking; normalized probabilities are only used as HMM emissions.

In [8]:
def scaled_pipeline(estimator):
    return Pipeline([
        ("preprocess", Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler())
        ])),
        ("model", estimator)
    ])


def tree_pipeline(estimator):
    return Pipeline([
        ("preprocess", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("model", estimator)
    ])


def positive_probability(model, X_data):
    proba = model.predict_proba(X_data)
    if hasattr(model, "named_steps"):
        classes = model.named_steps["model"].classes_
    else:
        classes = model.classes_
    return proba[:, list(classes).index(1)]


def ranking_metrics(frame):
    top1_correct = 0
    top3_correct = 0
    reciprocal_ranks = []

    for _, group in frame.groupby(["trip_id", "obs_idx"]):
        if int(group["is_match"].sum()) != 1:
            continue
        ranked = group.sort_values("raw_probability", ascending=False)
        rank = int(np.where(ranked["is_match"].to_numpy() == 1)[0][0] + 1)
        top1_correct += int(rank == 1)
        top3_correct += int(rank <= 3)
        reciprocal_ranks.append(1.0 / rank)

    if not reciprocal_ranks:
        return {"Top-1 Accuracy": np.nan, "Top-3 Recall": np.nan, "MRR": np.nan}
    n = len(reciprocal_ranks)
    return {
        "Top-1 Accuracy": top1_correct / n,
        "Top-3 Recall": top3_correct / n,
        "MRR": float(np.mean(reciprocal_ranks)),
    }


def evaluate(name, y_true, raw_probability, ranking_frame):
    y_pred = (raw_probability >= 0.5).astype(int)
    out = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "Log Loss": log_loss(y_true, raw_probability, labels=[0, 1]),
        "Confusion Matrix": confusion_matrix(y_true, y_pred, labels=[0, 1]),
    }
    try:
        out["ROC-AUC"] = roc_auc_score(y_true, raw_probability)
    except ValueError:
        out["ROC-AUC"] = np.nan
    try:
        out["PR-AUC / AP"] = average_precision_score(y_true, raw_probability)
    except ValueError:
        out["PR-AUC / AP"] = np.nan
    out.update(ranking_metrics(ranking_frame))
    return out


def predict_normalized(model, candidate_df):
    raw = positive_probability(model, candidate_df[FEATURE_COLUMNS])
    out = candidate_df[IDENTIFIERS].copy()
    out["raw_probability"] = raw
    out["probability"] = raw
    out["probability"] = out.groupby(["trip_id", "obs_idx"])["probability"].transform(
        lambda s: s / s.sum() if s.sum() > 0 else np.ones(len(s)) / len(s)
    )
    return out


In [9]:

# RECOVER TUNED PIPELINES
# Reusing the best parameters from completed RandomizedSearchCV

tuned_pipelines = {

    "Logistic Regression": scaled_pipeline(
        LogisticRegression(
            C=0.001,
            class_weight=None,
            max_iter=2000,
            solver="lbfgs",
            random_state=42
        )
    ),

    "Decision Tree": tree_pipeline(
        DecisionTreeClassifier(
            max_depth=8,
            min_samples_split=2,
            min_samples_leaf=30,
            class_weight=None,
            random_state=42
        )
    ),

    "Random Forest": tree_pipeline(
        RandomForestClassifier(
            n_estimators=100,
            max_depth=8,
            min_samples_leaf=3,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        )
    ),

    "Gradient Boosting": tree_pipeline(
        GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.03,
            max_depth=5,
            min_samples_leaf=10,
            random_state=42
        )
    ),

    "KNN": scaled_pipeline(
        KNeighborsClassifier(
            n_neighbors=31,
            weights="distance",
            metric="manhattan",
            n_jobs=-1
        )
    )
}

print("Successfully recovered tuned pipelines!")

for name in tuned_pipelines:
    print(name)


Successfully recovered tuned pipelines!
Logistic Regression
Decision Tree
Random Forest
Gradient Boosting
KNN


## 6. Baseline training

All five baselines use the exact same features and fold indices.

In [13]:
model_specs = {
    "Logistic Regression": scaled_pipeline(
        LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs", random_state=42)),
    "Decision Tree": tree_pipeline(
        DecisionTreeClassifier(random_state=42, class_weight="balanced")),
    "Random Forest": tree_pipeline(
        RandomForestClassifier(n_estimators=100, random_state=42,
                               class_weight="balanced_subsample", n_jobs=-1)),
    "Gradient Boosting": tree_pipeline(
        GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                   max_depth=3, random_state=42)),
    "KNN": scaled_pipeline(
        KNeighborsClassifier(n_neighbors=15, weights="distance",
                             metric="euclidean", n_jobs=-1)),
}

baseline_oof = {}
baseline_metrics = []

for name, pipeline in model_specs.items():
    print("Baseline:", name)
    oof = np.zeros(len(train), dtype=float)
    for train_idx, valid_idx in cv_splits:
        fold_model = clone(pipeline)
        fold_model.fit(X.iloc[train_idx], y.iloc[train_idx])
        oof[valid_idx] = positive_probability(fold_model, X.iloc[valid_idx])
    baseline_oof[name] = oof
    ranking_frame = train[["trip_id", "obs_idx", "is_match"]].copy()
    ranking_frame["raw_probability"] = oof
    baseline_metrics.append(evaluate(name, y.to_numpy(), oof, ranking_frame))

baseline_table = pd.DataFrame([
    {k: v for k, v in m.items() if k != "Confusion Matrix"}
    for m in baseline_metrics
]).sort_values(["Top-1 Accuracy", "MRR", "PR-AUC / AP"], ascending=False)
print("\nBaseline comparison:")
print(baseline_table.to_string(index=False))


Baseline: Logistic Regression
Baseline: Decision Tree
Baseline: Random Forest
Baseline: Gradient Boosting
Baseline: KNN

Baseline comparison:
              Model  Accuracy  Precision   Recall  Macro F1  Weighted F1  Log Loss  ROC-AUC  PR-AUC / AP  Top-1 Accuracy  Top-3 Recall      MRR
  Gradient Boosting  0.915035   0.786373 0.707115  0.846840     0.913230  0.205587 0.951794     0.818564        0.783329      0.972013 0.879166
      Random Forest  0.910335   0.776982 0.684737  0.837135     0.908064  0.291625 0.946207     0.803462        0.768970      0.969690 0.871339
Logistic Regression  0.806073   0.470231 0.844553  0.737846     0.824724  0.413394 0.898318     0.655529        0.759514      0.947437 0.857070
                KNN  0.899374   0.744664 0.647724  0.816326     0.896557  0.693371 0.915360     0.738731        0.733198      0.958281 0.848248
      Decision Tree  0.883468   0.675323 0.644878  0.794722     0.882402  4.200005 0.789510     0.497705        0.712240      0.960081 0.8

## 7. Time-bounded hyperparameter search

Optuna uses at most three trials and 240 seconds per model. If Optuna is not
available, `RandomizedSearchCV` uses the same `cv_splits` and three candidates.

In [14]:
TUNING_TRIALS = 3
TUNING_TIMEOUT_SECONDS = 240
tuned_pipelines = {}
search_summary = []

if OPTUNA_AVAILABLE:
    def tune(name, factory):
        def objective(trial):
            pipeline = factory(trial)
            scores = []
            for train_idx, valid_idx in cv_splits:
                fold_model = clone(pipeline)
                fold_model.fit(X.iloc[train_idx], y.iloc[train_idx])
                prob = positive_probability(fold_model, X.iloc[valid_idx])
                try:
                    scores.append(average_precision_score(y.iloc[valid_idx], prob))
                except ValueError:
                    scores.append(0.0)
            return float(np.mean(scores))
        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=TUNING_TRIALS,
                       timeout=TUNING_TIMEOUT_SECONDS)
        return study

    def lr_trial(t):
        return scaled_pipeline(LogisticRegression(
            C=t.suggest_float("C", 1e-3, 10, log=True),
            class_weight=t.suggest_categorical("class_weight", ["balanced", None]),
            max_iter=2000, solver="lbfgs", random_state=42))

    def dt_trial(t):
        return tree_pipeline(DecisionTreeClassifier(
            max_depth=t.suggest_int("max_depth", 2, 20),
            min_samples_split=t.suggest_int("min_samples_split", 2, 50),
            min_samples_leaf=t.suggest_int("min_samples_leaf", 1, 30),
            class_weight=t.suggest_categorical("class_weight", ["balanced", None]),
            random_state=42))

    def rf_trial(t):
        return tree_pipeline(RandomForestClassifier(
            n_estimators=t.suggest_int("n_estimators", 60, 160),
            max_depth=t.suggest_int("max_depth", 5, 25),
            min_samples_leaf=t.suggest_int("min_samples_leaf", 1, 10),
            class_weight=t.suggest_categorical(
                "class_weight", ["balanced", "balanced_subsample", None]),
            n_jobs=-1, random_state=42))

    def gb_trial(t):
        return tree_pipeline(GradientBoostingClassifier(
            n_estimators=t.suggest_int("n_estimators", 50, 150),
            learning_rate=t.suggest_float("learning_rate", 0.02, 0.2, log=True),
            max_depth=t.suggest_int("max_depth", 1, 5),
            min_samples_leaf=t.suggest_int("min_samples_leaf", 1, 30),
            random_state=42))

    def knn_trial(t):
        return scaled_pipeline(KNeighborsClassifier(
            n_neighbors=t.suggest_int("n_neighbors", 3, 51, step=2),
            weights=t.suggest_categorical("weights", ["uniform", "distance"]),
            metric=t.suggest_categorical("metric", ["euclidean", "manhattan"]),
            n_jobs=-1))

    factories = {
        "Logistic Regression": lr_trial, "Decision Tree": dt_trial,
        "Random Forest": rf_trial, "Gradient Boosting": gb_trial,
        "KNN": knn_trial
    }
    for name, factory in factories.items():
        print("Tuning:", name)
        study = tune(name, factory)
        if len(study.trials) == 0:
            tuned_pipelines[name] = clone(model_specs[name])
            continue
        best = study.best_params
        if name == "Logistic Regression":
            pipeline = scaled_pipeline(LogisticRegression(
                C=best["C"], class_weight=best["class_weight"],
                max_iter=2000, solver="lbfgs", random_state=42))
        elif name == "Decision Tree":
            pipeline = tree_pipeline(DecisionTreeClassifier(
                max_depth=best["max_depth"],
                min_samples_split=best["min_samples_split"],
                min_samples_leaf=best["min_samples_leaf"],
                class_weight=best["class_weight"], random_state=42))
        elif name == "Random Forest":
            pipeline = tree_pipeline(RandomForestClassifier(
                n_estimators=best["n_estimators"], max_depth=best["max_depth"],
                min_samples_leaf=best["min_samples_leaf"],
                class_weight=best["class_weight"], n_jobs=-1, random_state=42))
        elif name == "Gradient Boosting":
            pipeline = tree_pipeline(GradientBoostingClassifier(
                n_estimators=best["n_estimators"],
                learning_rate=best["learning_rate"], max_depth=best["max_depth"],
                min_samples_leaf=best["min_samples_leaf"], random_state=42))
        else:
            pipeline = scaled_pipeline(KNeighborsClassifier(
                n_neighbors=best["n_neighbors"], weights=best["weights"],
                metric=best["metric"], n_jobs=-1))
        tuned_pipelines[name] = pipeline
        search_summary.append({"Model": name, "Best CV AP": study.best_value,
                               "Best parameters": best})
else:
    print("Optuna unavailable: using RandomizedSearchCV")
    searches = {
        "Logistic Regression": (
            clone(model_specs["Logistic Regression"]),
            {"model__C": [0.001, 0.01, 0.1, 1.0, 10.0],
             "model__class_weight": ["balanced", None]}),
        "Decision Tree": (
            clone(model_specs["Decision Tree"]),
            {"model__max_depth": [4, 8, 12, 20, None],
             "model__min_samples_split": [2, 10, 30, 50],
             "model__min_samples_leaf": [1, 5, 15, 30],
             "model__class_weight": ["balanced", None]}),
        "Random Forest": (
            clone(model_specs["Random Forest"]),
            {"model__n_estimators": [60, 100, 160],
             "model__max_depth": [8, 12, 20, None],
             "model__min_samples_leaf": [1, 3, 7],
             "model__class_weight": ["balanced", "balanced_subsample", None]}),
        "Gradient Boosting": (
            clone(model_specs["Gradient Boosting"]),
            {"model__n_estimators": [50, 100, 150],
             "model__learning_rate": [0.03, 0.1, 0.2],
             "model__max_depth": [1, 3, 5],
             "model__min_samples_leaf": [1, 10, 30]}),
        "KNN": (
            clone(model_specs["KNN"]),
            {"model__n_neighbors": [3, 7, 15, 31, 51],
             "model__weights": ["uniform", "distance"],
             "model__metric": ["euclidean", "manhattan"]}),
    }
    for name, (pipeline, params) in searches.items():
        print("Randomized tuning:", name)
        search = RandomizedSearchCV(
            pipeline, params, n_iter=TUNING_TRIALS,
            scoring="average_precision", cv=cv_splits,
            n_jobs=-1, random_state=42, refit=True)
        search.fit(X, y)
        tuned_pipelines[name] = search.best_estimator_
        search_summary.append({"Model": name, "Best CV AP": search.best_score_,
                               "Best parameters": search.best_params_})

print(pd.DataFrame(search_summary).to_string(index=False))


Optuna unavailable: using RandomizedSearchCV
Randomized tuning: Logistic Regression
Randomized tuning: Decision Tree
Randomized tuning: Random Forest
Randomized tuning: Gradient Boosting
Randomized tuning: KNN
              Model  Best CV AP                                                                                                      Best parameters
Logistic Regression    0.683335                                                                     {'model__class_weight': None, 'model__C': 0.001}
      Decision Tree    0.808777   {'model__min_samples_split': 2, 'model__min_samples_leaf': 30, 'model__max_depth': 8, 'model__class_weight': None}
      Random Forest    0.835297 {'model__n_estimators': 100, 'model__min_samples_leaf': 3, 'model__max_depth': 8, 'model__class_weight': 'balanced'}
  Gradient Boosting    0.833522     {'model__n_estimators': 100, 'model__min_samples_leaf': 10, 'model__max_depth': 5, 'model__learning_rate': 0.03}
                KNN    0.770467              

## 8. Tuned OOF comparison and best-model selection

The model with the highest OOF top-1 candidate accuracy is selected; MRR
and PR-AUC break ties.

In [15]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier

# Gradient Boosting pipeline (FIXED)
gb_pipeline = tree_pipeline(
    GradientBoostingClassifier(random_state=42)
)

gb_params = {
    "model__n_estimators": [50, 100, 150],
    "model__learning_rate": [0.03, 0.1, 0.2],
    "model__max_depth": [1, 3, 5],
    "model__min_samples_leaf": [1, 10, 30]
}

gb_search = RandomizedSearchCV(
    estimator=gb_pipeline,
    param_distributions=gb_params,
    n_iter=3,
    scoring="average_precision",
    cv=cv_splits,
    n_jobs=-1,
    random_state=42,
    refit=True
)

print("Tuning Gradient Boosting...")

gb_search.fit(X, y)

tuned_gb = gb_search.best_estimator_

print("\nBest CV AP:", gb_search.best_score_)
print("\nBest Parameters:")
print(gb_search.best_params_)

tuned_pipelines["Gradient Boosting"] = tuned_gb

Tuning Gradient Boosting...

Best CV AP: 0.8335216701367862

Best Parameters:
{'model__n_estimators': 100, 'model__min_samples_leaf': 10, 'model__max_depth': 5, 'model__learning_rate': 0.03}


## 9. Visualizations

The following figures show model ranking, probabilistic quality, ROC/PR curves,
and tuned-model confusion matrices.

In [11]:
ranking_names = ["Top-1 Accuracy", "Top-3 Recall", "MRR", "PR-AUC / AP"]
comparison.set_index("Model")[ranking_names].plot(
    kind="bar", figsize=(12, 6), rot=25)
plt.title("Tuned model comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

class_names = ["Accuracy", "Precision", "Recall", "Macro F1",
               "Weighted F1", "ROC-AUC", "PR-AUC / AP"]
plt.figure(figsize=(12, 5))
sns.heatmap(comparison.set_index("Model")[class_names],
            annot=True, fmt=".3f", cmap="viridis", vmin=0, vmax=1)
plt.title("Tuned classification metrics")
plt.tight_layout()
plt.show()
print(comparison[["Model", "Log Loss"]].sort_values("Log Loss").to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, oof in tuned_oof.items():
    fpr, tpr, _ = roc_curve(y, oof)
    precision, recall, _ = precision_recall_curve(y, oof)
    axes[0].plot(fpr, tpr, label=name)
    axes[1].plot(recall, precision, label=name)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[0].set_title("OOF ROC curves"); axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate"); axes[0].legend(fontsize=8)
axes[1].set_title("OOF Precision-Recall curves"); axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision"); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

n = len(tuned_metrics)
fig, axes = plt.subplots(int(np.ceil(n / 3)), 3, figsize=(14, 4 * int(np.ceil(n / 3))))
axes = np.atleast_1d(axes).ravel()
for ax, metrics in zip(axes, tuned_metrics):
    ConfusionMatrixDisplay(
        confusion_matrix=metrics["Confusion Matrix"], display_labels=[0, 1]
    ).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(metrics["Model"])
for ax in axes[n:]:
    ax.axis("off")
plt.tight_layout()
plt.show()


NameError: name 'comparison' is not defined

## 10. Final held-out student test evaluation

The selected model is not refitted or tuned on this data.

In [12]:
test_predictions = predict_normalized(best_model, train_test)
test_ranking = train_test[["trip_id", "obs_idx", "is_match"]].copy()
test_ranking["raw_probability"] = test_predictions["raw_probability"].to_numpy()
test_ranking_metrics = ranking_metrics(test_ranking)

test_pred = test_predictions["raw_probability"].to_numpy()
test_class = (test_pred >= 0.5).astype(int)
print("=== Student test held-out evaluation ===")
print("Accuracy:", accuracy_score(y_test, test_class))
print("Precision:", precision_score(y_test, test_class, zero_division=0))
print("Recall:", recall_score(y_test, test_class, zero_division=0))
print("Macro F1:", f1_score(y_test, test_class, average="macro", zero_division=0))
print("Weighted F1:", f1_score(y_test, test_class, average="weighted", zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, test_pred))
print("PR-AUC / Average Precision:", average_precision_score(y_test, test_pred))
print("Log Loss:", log_loss(y_test, test_pred, labels=[0, 1]))
print("Top-1 Accuracy:", test_ranking_metrics["Top-1 Accuracy"])
print("Top-3 Recall:", test_ranking_metrics["Top-3 Recall"])
print("MRR:", test_ranking_metrics["MRR"])
print("Normalized probability sum description:")
print(test_predictions.groupby(["trip_id", "obs_idx"])["probability"].sum().describe().to_string())


NameError: name 'best_model' is not defined

## 11. Export for integration

The export includes the sklearn pipeline, metadata, candidate predictions,
and model comparison. `probability` sums to 1 within each
`(trip_id, obs_idx)` group.

In [13]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

model_path = ARTIFACT_DIR / "pathsnap_best_model.joblib"
metadata_path = ARTIFACT_DIR / "pathsnap_model_metadata.json"
predictions_path = ARTIFACT_DIR / "student_test_candidate_predictions.csv"
comparison_path = ARTIFACT_DIR / "pathsnap_tuned_model_comparison.csv"

joblib.dump(best_model, model_path)
test_predictions.to_csv(predictions_path, index=False)
comparison.to_csv(comparison_path, index=False)

metadata = {
    "selected_model": best_name,
    "feature_columns": FEATURE_COLUMNS,
    "target": TARGET,
    "identifier_columns": IDENTIFIERS,
    "cv_strategy": "GroupKFold",
    "cv_groups": "trip_id",
    "n_splits": n_splits,
    "training_rows": int(len(train)),
    "training_trips": int(groups.nunique()),
    "student_test_rows": int(len(train_test)),
    "probability_format": "raw_probability is binary positive-class probability; probability is normalized within (trip_id, obs_idx)",
    "student_test_ranking": test_ranking_metrics,
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(model_path)
print(metadata_path)
print(predictions_path)
print(comparison_path)

try:
    from google.colab import files
    for path in [model_path, metadata_path, predictions_path, comparison_path]:
        files.download(str(path))
except ImportError:
    pass


NameError: name 'best_model' is not defined

## 12. Reusable candidate probability function

In [ ]:
def predict_candidate_probabilities(model, candidate_df):
    # raw_probability is binary positive-class probability.
    # probability sums to 1 within every (trip_id, obs_idx) observation.
    required = set(FEATURE_COLUMNS + IDENTIFIERS)
    missing = sorted(required - set(candidate_df.columns))
    if missing:
        raise ValueError(f"Candidate table missing columns: {missing}")
    return predict_normalized(model, candidate_df)


loaded_model = joblib.load(model_path)
example_predictions = predict_candidate_probabilities(
    loaded_model, train_test.head(100).copy())
example_predictions.head(10)


## 13. Final summary

- Exactly five required model families were trained and tuned.
- Every model used the same feature set and same grouped fold indices.
- Ranking metrics use `(trip_id, obs_idx)` and raw probabilities.
- The student test set was held out from fitting and tuning.
- The exported `probability` field is normalized for HMM emissions.

In [ ]:
summary = pd.DataFrame([{
    "Selected model": best_name,
    "CV": f"{n_splits}-fold GroupKFold by trip_id",
    "Training rows": len(train),
    "Training trips": groups.nunique(),
    "OOF Top-1": comparison.iloc[0]["Top-1 Accuracy"],
    "OOF Top-3": comparison.iloc[0]["Top-3 Recall"],
    "OOF MRR": comparison.iloc[0]["MRR"],
    "Student test Top-1": test_ranking_metrics["Top-1 Accuracy"],
    "Student test Top-3": test_ranking_metrics["Top-3 Recall"],
    "Student test MRR": test_ranking_metrics["MRR"],
}])
summary
